# Stage 1: BatDetect2 Inference → Detection CSV

Run BatDetect2 inference **once** on all audio files and save per-detection
results (metadata + raw 32-dim CNN features) to CSV. This is the slow step
(GPU-bound). Stage 2 notebooks then read this CSV to generate different
embedding databases quickly without re-running inference.

## Outputs
- **`batdetect2_detections.csv`**: One row per detection with metadata + 32 raw feature columns
- **`batdetect2_file_index.csv`**: One row per audio file with duration and detection count

## Stage 2 notebooks (read from CSV)
- `embed_32dim.ipynb` → per-detection DB (one embedding per detection)
- `embed_32dim_avg.ipynb` → window-averaged DB (one embedding per 0.5s window)

## 1. Environment Setup

In [ ]:
# Optional: Disable TensorFlow GPU and XLA for CPU-only environments
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [4]:
# Core imports
import os
import csv
import time
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
from tqdm import tqdm

# BatDetect2
from batdetect2 import api
from batdetect2.utils import detector_utils as du

print('Imports OK!')
try:
    import batdetect2
    print(f'batdetect2: {getattr(batdetect2, "__version__", "unknown")}')
except Exception:
    print('batdetect2: imported (version unknown)')

Imports OK!
batdetect2: 1.3.0


## 2. Configuration

In [5]:
# =============================================================================
# USER CONFIGURATION
# =============================================================================

# Path to audio files (NOT time-expanded)
DATASET_BASE_PATH = "C:/Users/Administrator/hello/uk_bats/uk_audio1"

# Output CSV paths
OUTPUT_DIR = "C:/Users/Administrator/hello/uk_bats/uk_audio"
DETECTIONS_CSV = os.path.join(OUTPUT_DIR, "batdetect2_detections.csv")
FILE_INDEX_CSV = os.path.join(OUTPUT_DIR, "batdetect2_file_index.csv")

# File pattern
FILE_GLOB = "*.wav"

# =============================================================================
# BATDETECT2 CONFIGURATION
# =============================================================================

# Audio is NOT time-expanded
TIME_EXPANSION = 1.0

# Detection threshold
DETECTION_THRESHOLD = 0.3

# BatDetect2's native sample rate
TARGET_SAMP_RATE = 256000

# Chunk size for processing long files
CHUNK_SIZE = 3.0

# Embedding dimension (32-dim CNN features)
EMBEDDING_DIM = 32

# Get BatDetect2 processing config
bd2_config = api.get_config(
    detection_threshold=DETECTION_THRESHOLD,
    time_expansion=TIME_EXPANSION,
    target_samp_rate=TARGET_SAMP_RATE,
    chunk_size=CHUNK_SIZE,
    cnn_features=True,
    spec_features=False,
    spec_slices=False,
)

print("Configuration loaded.")
print(f"  Audio path: {DATASET_BASE_PATH}")
print(f"  Detections CSV: {DETECTIONS_CSV}")
print(f"  File index CSV: {FILE_INDEX_CSV}")
print(f"  Time expansion: {TIME_EXPANSION}")
print(f"  Detection threshold: {DETECTION_THRESHOLD}")
print(f"  Chunk size: {CHUNK_SIZE}s")
print(f"  Embedding dim: {EMBEDDING_DIM}")

Configuration loaded.
  Audio path: C:/Users/Administrator/hello/uk_bats/uk_audio1
  Detections CSV: C:/Users/Administrator/hello/uk_bats/uk_audio\batdetect2_detections.csv
  File index CSV: C:/Users/Administrator/hello/uk_bats/uk_audio\batdetect2_file_index.csv
  Time expansion: 1.0
  Detection threshold: 0.3
  Chunk size: 3.0s
  Embedding dim: 32


## 3. Single File Test

In [6]:
# Find audio files
audio_dir = Path(DATASET_BASE_PATH)
wav_files = sorted(audio_dir.glob(FILE_GLOB))
print(f"Found {len(wav_files)} audio files")

# Test with one file
test_path = wav_files[0]
print(f"\nTest file: {test_path.name}")

audio_full = api.load_audio(
    str(test_path),
    time_exp_fact=TIME_EXPANSION,
    target_samp_rate=TARGET_SAMP_RATE,
)
duration_s = len(audio_full) / TARGET_SAMP_RATE
print(f"  Duration: {duration_s:.3f}s ({len(audio_full)} samples at {TARGET_SAMP_RATE} Hz)")

# Process in chunks
n_det_total = 0
for chunk_time, audio_chunk in du.iterate_over_chunks(audio_full, TARGET_SAMP_RATE, CHUNK_SIZE):
    detections, features, spec = api.process_audio(
        audio_chunk, samp_rate=TARGET_SAMP_RATE, config=bd2_config
    )
    n_det = len(detections)
    n_det_total += n_det
    print(f"  Chunk at {chunk_time:.2f}s: {n_det} detections")
    if n_det > 0:
        print(f"    Features shape: {features.shape}")
        print(f"    Feature range: [{features.min():.6f}, {features.max():.6f}]")
        det = detections[0]
        print(f"    First detection fields: {list(det.keys())}")
        print(f"    start_time={det['start_time']:.4f}, end_time={det['end_time']:.4f}")
        print(f"    low_freq={det['low_freq']}, high_freq={det['high_freq']}")
        print(f"    class='{det['class']}', class_prob={det['class_prob']:.4f}, det_prob={det['det_prob']:.4f}")

print(f"\nTotal detections: {n_det_total}")

Found 9997 audio files

Test file: 00000_myotis_alcathoe_0_1.0.wav
  Duration: 1.000s (256001 samples at 256000 Hz)
  Chunk at 0.00s: 18 detections
    Features shape: (18, 32)
    Feature range: [0.000000, 6.637173]
    First detection fields: ['start_time', 'end_time', 'low_freq', 'high_freq', 'class', 'class_prob', 'det_prob', 'individual', 'event']
    start_time=0.0385, end_time=0.0424
    low_freq=27187, high_freq=86784
    class='Myotis nattereri', class_prob=0.3830, det_prob=0.4630

Total detections: 18


## 4. Batch Processing Function

In [7]:
# CSV column names
FEAT_COLS = [f"feat_{i}" for i in range(EMBEDDING_DIM)]
DET_COLUMNS = [
    "source_id", "chunk_offset_s",
    "det_start_time", "det_end_time",
    "low_freq", "high_freq",
    "class", "class_prob", "det_prob",
] + FEAT_COLS

INDEX_COLUMNS = ["source_id", "duration_s", "n_detections", "n_chunks"]


def run_stage1(
    audio_dir: str,
    config: dict,
    detections_csv: str,
    file_index_csv: str,
    time_expansion: float = 1.0,
    target_samp_rate: int = 256000,
    chunk_size: float = 3.0,
    file_glob: str = "*.wav",
) -> Dict:
    """Run BatDetect2 inference on all audio files and save results to CSV.

    Writes incrementally to avoid holding all features in memory.

    Returns:
        Dict with summary statistics.
    """
    audio_path = Path(audio_dir)
    wav_files = sorted(audio_path.glob(file_glob))
    n_files = len(wav_files)
    print(f"Processing {n_files} audio files...")

    # Stats
    stats = {
        "files_processed": 0,
        "files_with_detections": 0,
        "files_without_detections": 0,
        "total_detections": 0,
        "total_chunks": 0,
        "errors": 0,
        "species_counts": {},
    }

    # Open CSV files for writing
    with open(detections_csv, "w", newline="") as det_f, \
         open(file_index_csv, "w", newline="") as idx_f:

        det_writer = csv.writer(det_f)
        det_writer.writerow(DET_COLUMNS)

        idx_writer = csv.writer(idx_f)
        idx_writer.writerow(INDEX_COLUMNS)

        for wav_path in tqdm(wav_files, desc="Stage 1 inference"):
            source_id = wav_path.name

            try:
                # Load and resample audio
                audio_full = api.load_audio(
                    str(wav_path),
                    time_exp_fact=time_expansion,
                    target_samp_rate=target_samp_rate,
                )
                duration_s = len(audio_full) / target_samp_rate

                file_n_det = 0
                file_n_chunks = 0

                # Process in chunks (avoids features[0] bug in process_file)
                for chunk_time, audio_chunk in du.iterate_over_chunks(
                    audio_full, target_samp_rate, chunk_size
                ):
                    file_n_chunks += 1
                    stats["total_chunks"] += 1

                    detections, features, spec = api.process_audio(
                        audio_chunk,
                        samp_rate=target_samp_rate,
                        config=config,
                    )

                    if len(detections) == 0:
                        continue

                    # Write each detection as a CSV row
                    for i, det in enumerate(detections):
                        abs_start = det["start_time"] + chunk_time
                        abs_end = det["end_time"] + chunk_time
                        feat_values = features[i].tolist()

                        row = [
                            source_id,
                            f"{chunk_time:.6f}",
                            f"{abs_start:.6f}",
                            f"{abs_end:.6f}",
                            det["low_freq"],
                            det["high_freq"],
                            det["class"],
                            f"{det['class_prob']:.6f}",
                            f"{det['det_prob']:.6f}",
                        ] + [f"{v:.8f}" for v in feat_values]

                        det_writer.writerow(row)
                        file_n_det += 1

                        # Track species
                        sp = det["class"]
                        stats["species_counts"][sp] = stats["species_counts"].get(sp, 0) + 1

                # Write file index row
                idx_writer.writerow([source_id, f"{duration_s:.6f}", file_n_det, file_n_chunks])

                stats["files_processed"] += 1
                stats["total_detections"] += file_n_det
                if file_n_det > 0:
                    stats["files_with_detections"] += 1
                else:
                    stats["files_without_detections"] += 1

            except Exception as e:
                print(f"\nError processing {source_id}: {e}")
                stats["errors"] += 1
                # Still write an index entry with 0 detections
                idx_writer.writerow([source_id, "0.0", 0, 0])

    return stats


print("run_stage1() defined")
print(f"Detection CSV columns ({len(DET_COLUMNS)}): {DET_COLUMNS[:9]} + feat_0..feat_31")
print(f"File index columns: {INDEX_COLUMNS}")

run_stage1() defined
Detection CSV columns (41): ['source_id', 'chunk_offset_s', 'det_start_time', 'det_end_time', 'low_freq', 'high_freq', 'class', 'class_prob', 'det_prob'] + feat_0..feat_31
File index columns: ['source_id', 'duration_s', 'n_detections', 'n_chunks']


## 5. Run Batch Processing

In [8]:
start_time = time.time()

stats = run_stage1(
    audio_dir=DATASET_BASE_PATH,
    config=bd2_config,
    detections_csv=DETECTIONS_CSV,
    file_index_csv=FILE_INDEX_CSV,
    time_expansion=TIME_EXPANSION,
    target_samp_rate=TARGET_SAMP_RATE,
    chunk_size=CHUNK_SIZE,
    file_glob=FILE_GLOB,
)

elapsed = time.time() - start_time
print(f"\nDone in {elapsed:.1f}s ({elapsed/60:.1f} min)")

Processing 9997 audio files...


Stage 1 inference:  57%|████████████████████████████████▍                        | 5682/9997 [2:44:58<30:46,  2.34it/s]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



## 6. Summary Statistics

In [9]:
print("=" * 60)
print("STAGE 1 SUMMARY")
print("=" * 60)
print(f"Files processed:           {stats['files_processed']}")
print(f"Files with detections:     {stats['files_with_detections']}")
print(f"Files without detections:  {stats['files_without_detections']}")
print(f"Total chunks processed:    {stats['total_chunks']}")
print(f"Total detections:          {stats['total_detections']}")
print(f"Errors:                    {stats['errors']}")
print(f"\nDetections CSV: {DETECTIONS_CSV}")
print(f"File index CSV: {FILE_INDEX_CSV}")

# CSV file sizes
det_size = os.path.getsize(DETECTIONS_CSV) / (1024 * 1024)
idx_size = os.path.getsize(FILE_INDEX_CSV) / (1024 * 1024)
print(f"\nDetections CSV size: {det_size:.1f} MB")
print(f"File index CSV size: {idx_size:.2f} MB")

# Species distribution
print(f"\nSpecies distribution ({len(stats['species_counts'])} species):")
for sp, count in sorted(stats['species_counts'].items(), key=lambda x: -x[1]):
    pct = 100 * count / stats['total_detections'] if stats['total_detections'] > 0 else 0
    print(f"  {sp:40s}  {count:6d}  ({pct:.1f}%)")

STAGE 1 SUMMARY
Files processed:           9997
Files with detections:     9238
Files without detections:  759
Total chunks processed:    15083
Total detections:          88469
Errors:                    0

Detections CSV: C:/Users/Administrator/hello/uk_bats/uk_audio\batdetect2_detections.csv
File index CSV: C:/Users/Administrator/hello/uk_bats/uk_audio\batdetect2_file_index.csv

Detections CSV size: 39.2 MB
File index CSV size: 0.46 MB

Species distribution (17 species):
  Pipistrellus pipistrellus                  14563  (16.5%)
  Nyctalus noctula                           13163  (14.9%)
  Eptesicus serotinus                        10223  (11.6%)
  Myotis daubentonii                          7520  (8.5%)
  Myotis nattereri                            7140  (8.1%)
  Pipistrellus pygmaeus                       6532  (7.4%)
  Pipistrellus nathusii                       5220  (5.9%)
  Myotis mystacinus                           4916  (5.6%)
  Nyctalus leisleri                           4

## 7. Sanity Checks

In [ ]:
# Load and inspect the detections CSV
df = pd.read_csv(DETECTIONS_CSV)
print(f"Detections CSV: {len(df)} rows x {len(df.columns)} columns")
print(f"\nColumn dtypes:")
print(df.dtypes)
print(f"\nFirst 5 rows (metadata only):")
print(df[["source_id", "det_start_time", "det_end_time", "class", "class_prob", "det_prob"]].head())

In [ ]:
# Check feature value ranges
feat_cols = [f"feat_{i}" for i in range(EMBEDDING_DIM)]
feat_data = df[feat_cols].values

print(f"Feature matrix shape: {feat_data.shape}")
print(f"Feature value range: [{feat_data.min():.6f}, {feat_data.max():.6f}]")
print(f"Feature mean: {feat_data.mean():.6f}")
print(f"Feature std: {feat_data.std():.6f}")
print(f"Any NaN: {np.isnan(feat_data).any()}")
print(f"Any Inf: {np.isinf(feat_data).any()}")
print(f"Any negative: {(feat_data < 0).any()}")
print(f"Fraction zero: {(feat_data == 0).mean():.3f}")

# Per-dimension stats
print(f"\nPer-dimension ranges:")
for i in range(EMBEDDING_DIM):
    col = feat_data[:, i]
    print(f"  feat_{i:2d}: min={col.min():.6f}  max={col.max():.6f}  mean={col.mean():.6f}")

In [ ]:
# Check file index
idx_df = pd.read_csv(FILE_INDEX_CSV)
print(f"File index: {len(idx_df)} files")
print(f"\nDuration stats:")
print(idx_df["duration_s"].describe())
print(f"\nDetection count stats:")
print(idx_df["n_detections"].describe())
print(f"\nFiles with 0 detections: {(idx_df['n_detections'] == 0).sum()}")
print(f"Files with >10 detections: {(idx_df['n_detections'] > 10).sum()}")

# Cross-check: total detections in index vs CSV rows
total_from_index = idx_df["n_detections"].sum()
total_from_csv = len(df)
print(f"\nCross-check:")
print(f"  Total detections from file index: {total_from_index}")
print(f"  Total rows in detections CSV:     {total_from_csv}")
print(f"  Match: {total_from_index == total_from_csv}")

## 8. Summary

Stage 1 complete. The CSVs are now ready for Stage 2 processing:
- `embed_32dim.ipynb` → reads CSV, creates per-detection hoplite DB
- `embed_32dim_avg.ipynb` → reads CSV + file index, creates window-averaged hoplite DB